In [1]:
import pandas as pd

***1. Dates en datetime et en heure de Paris, identifiants en texte, états en `category`.***

In [2]:
df = pd.read_csv("./data/raw/comptages.csv", sep=";") 

# Afficher les premières lignes du dataset
df.head()

,iu_ac,libelle,t_1h,q,k,etat_trafic,iu_nd_amont,libelle_nd_amont,iu_nd_aval,libelle_nd_aval,etat_barre
0,5871,Bd_Macdonald,2026-08-31T23:00:00+00:00,94.0,NaN,Inconnu,3048,Bd Macdonald-Allee des Cardinoux,928,Bd_Macdonald - Jaques Duchesne,Ouvert
1,1744,Bd_Macdonald,2026-08-31T23:00:00+00:00,94.0,0.52500,Fluide,928,Bd_Macdonald - Jaques Duchesne,929,Bd_Macdonald - Lounes Matoub,Ouvert
2,1745,Bd_Macdonald,2026-08-31T23:00:00+00:00,NaN,0.51833,Fluide,930,Bd_Macdonald - Chana Orloff,929,Bd_Macdonald - Lounes Matoub,Ouvert
3,1746,Bd_Macdonald,2026-08-31T23:00:00+00:00,48.0,NaN,Inconnu,929,Bd_Macdonald - Lounes Matoub,3630,Bd Macdonald - Rue E. 019,Ouvert
4,1747,Bd_Macdonald,2026-08-31T23:00:00+00:00,NaN,NaN,Inconnu,3630,Bd Macdonald - Rue E. 019,930,Bd_Macdonald - Chana Orloff,Invalide


In [3]:
# Mettre 'Dates' en datetime
df['t_1h'] = pd.to_datetime(df['t_1h'])

# Rajouter une colonne pour mettre en heure de Paris
df['Dates_Paris'] = df['t_1h'].dt.tz_convert('Europe/Paris')

In [4]:
# Cibler la vraie nuit du changement d'heure (29 mars 2026)
mask_transition = (df['t_1h'] >= '2026-03-29 00:00:00') & (df['t_1h'] <= '2026-03-29 02:00:00')

df_transition = df.loc[mask_transition].copy()
df_transition['decalage'] = (df_transition['Dates_Paris'].dt.hour - df_transition['t_1h'].dt.hour) % 24

# Nettoyer l'affichage pour ne voir qu'une ligne par horaire
affichage = df_transition[['t_1h', 'Dates_Paris', 'decalage']].drop_duplicates(subset=['t_1h'])

df.drop(columns=['t_1h'], inplace=True)
print(affichage)

                            t_1h               Dates_Paris  decalage
170798 2026-03-29 02:00:00+00:00 2026-03-29 04:00:00+02:00         2
170844 2026-03-29 01:00:00+00:00 2026-03-29 03:00:00+02:00         2
170890 2026-03-29 00:00:00+00:00 2026-03-29 01:00:00+01:00         1


In [5]:
# Changer le type de la colonne 'libelle' en texte 
df['libelle'].astype(str)

# Changer le type de la colonne 'etat_trafic' en category
df['etat_trafic'].astype('category')

# Changer le type de la colonne 'etat_barre' en category
df['etat_barre'].astype('category')

df.head()

,iu_ac,libelle,q,k,etat_trafic,iu_nd_amont,libelle_nd_amont,iu_nd_aval,libelle_nd_aval,etat_barre,Dates_Paris
0,5871,Bd_Macdonald,94.0,NaN,Inconnu,3048,Bd Macdonald-Allee des Cardinoux,928,Bd_Macdonald - Jaques Duchesne,Ouvert,2026-09-01 01:00:00+02:00
1,1744,Bd_Macdonald,94.0,0.52500,Fluide,928,Bd_Macdonald - Jaques Duchesne,929,Bd_Macdonald - Lounes Matoub,Ouvert,2026-09-01 01:00:00+02:00
2,1745,Bd_Macdonald,NaN,0.51833,Fluide,930,Bd_Macdonald - Chana Orloff,929,Bd_Macdonald - Lounes Matoub,Ouvert,2026-09-01 01:00:00+02:00
3,1746,Bd_Macdonald,48.0,NaN,Inconnu,929,Bd_Macdonald - Lounes Matoub,3630,Bd Macdonald - Rue E. 019,Ouvert,2026-09-01 01:00:00+02:00
4,1747,Bd_Macdonald,NaN,NaN,Inconnu,3630,Bd Macdonald - Rue E. 019,930,Bd_Macdonald - Chana Orloff,Invalide,2026-09-01 01:00:00+02:00


***2. Doublons : testez sur la bonne clé, écrivez le résultat.***

In [6]:
doublons = df.duplicated(subset=["iu_ac", "Dates_Paris"]).sum()
print(f"{doublons} doublons sur la clé (iu_ac, t_1h)")

0 doublons sur la clé (iu_ac, t_1h)


In [7]:
# Nombre de capteurs et d'heures distincts
print(f"{df['iu_ac'].nunique()} capteurs distincts")
print(f"{df['Dates_Paris'].nunique()} heures distinctes")

# Doublons si on teste chaque colonne seule (pour comparaison)
print(f"{df.duplicated(subset=['iu_ac']).sum()} doublons sur iu_ac seul")
print(f"{df.duplicated(subset=['Dates_Paris']).sum()} doublons sur t_1h seul")

46 capteurs distincts
4364 heures distinctes
200698 doublons sur iu_ac seul
196380 doublons sur t_1h seul


In [8]:
pivot = df.pivot_table(index="Dates_Paris", columns="iu_ac", values="q")

capteurs_q_identiques = pivot.T[pivot.T.duplicated(keep=False)]
print(f"{capteurs_q_identiques.shape[0]} capteurs ont exactement la même série de débit qu'un autre")
print(capteurs_q_identiques.index.tolist())

32 capteurs ont exactement la même série de débit qu'un autre
[99, 100, 211, 212, 635, 636, 768, 782, 847, 1741, 1742, 1743, 1744, 1746, 1748, 1759, 4810, 4812, 4815, 4818, 4822, 4823, 4824, 4825, 4826, 4827, 4828, 4829, 5046, 5048, 5871, 5872]


***3. Colonnes supprimées : le test qui justifie chaque suppression.***

In [9]:
colonnes_a_supprimer = [
    'iu_ac',
    'iu_nd_amont',
    'iu_nd_aval'
]

# Suppression des colonnes
df.drop(columns=colonnes_a_supprimer, inplace=True)

'iu_ac' : Cette colonne correspond à l'identifiant du trouçon sous forme de code, elle fait un peu doublon avec la colonne 'libellé'. C'est pour cela que je l'ai supprimé. 

'iu_nd_amount' & 'iu_nd_aval' : Ces colonnes représentent les carrefours situés en amont et en aval du tronçon. Elles font doublons avec les colonnes 'libelle_nd_amont' et 'libelle_nd_aval', qui fournissent la même information.

In [10]:
df.head()

,libelle,q,k,etat_trafic,libelle_nd_amont,libelle_nd_aval,etat_barre,Dates_Paris
0,Bd_Macdonald,94.0,NaN,Inconnu,Bd Macdonald-Allee des Cardinoux,Bd_Macdonald - Jaques Duchesne,Ouvert,2026-09-01 01:00:00+02:00
1,Bd_Macdonald,94.0,0.52500,Fluide,Bd_Macdonald - Jaques Duchesne,Bd_Macdonald - Lounes Matoub,Ouvert,2026-09-01 01:00:00+02:00
2,Bd_Macdonald,NaN,0.51833,Fluide,Bd_Macdonald - Chana Orloff,Bd_Macdonald - Lounes Matoub,Ouvert,2026-09-01 01:00:00+02:00
3,Bd_Macdonald,48.0,NaN,Inconnu,Bd_Macdonald - Lounes Matoub,Bd Macdonald - Rue E. 019,Ouvert,2026-09-01 01:00:00+02:00
4,Bd_Macdonald,NaN,NaN,Inconnu,Bd Macdonald - Rue E. 019,Bd_Macdonald - Chana Orloff,Invalide,2026-09-01 01:00:00+02:00


***4. Valeurs manquantes : que veulent dire *Barré* et *Invalide* **sur vos axes** ? Quelle est la longueur de **vos** trous ? Choisissez votre méthode à partir de ces chiffres.***

In [11]:
df["etat_barre"].value_counts(dropna=False)

etat_barre
Ouvert      130764
Invalide     67978
Barré         2002
Name: count, dtype: int64

In [12]:
df.groupby("etat_barre")["q"].aggregate(["count", "mean", "min", "max"])

,count,mean,min,max
etat_barre,,,,
Barré,1,66.000000,66.0,66.00
Invalide,0,NaN,NaN,NaN
Ouvert,112708,186.484507,0.0,5036.95


'Barré', on remarque que un seul passage

'Invalide', on remarque 0 passage, cela signifie peut-être que la route est fermé à tous types de circulation.

In [13]:
trou = df.sort_values(["libelle", "Dates_Paris"])["q"].isna()
nouveau_groupe = (trou != trou.shift()) | (df["libelle"] != df["libelle"].shift())
longueurs = trou[trou].groupby(nouveau_groupe.cumsum()[trou]).size()
longueurs.describe()

count    47897.000000
mean         1.838007
std          1.359767
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max         15.000000
Name: q, dtype: float64

***5. Données propres dans `data/processed/` (qui n'est pas versionné).***